# SafeScan — Notebook 4: Novelty Additions
### Confidence Calibration + RAG Grounding + Standardized Nutrition Scoring

Re-synced to your current Notebook 3 pipeline (front image -> OCR + ResNet50 -> classify; back image -> OCR -> Gemini). Builds on top, doesn't modify Notebook 2 or 3.

| Addition | What it does | Answers panel question |
|---|---|---|
| **Part A — Confidence Calibration** | Mondrian conformal prediction on the classifier | "How do you know your confidence numbers mean anything?" |
| **Part B — RAG Grounding** | Retrieves real Open Food Facts data before the allergen prompt | "You're just calling Gemini API" |
| **Part C — Standardized Nutrition Scoring** (new) | Computes a nutrition grade using the **published Nutri-Score algorithm** (Sante publique France, EU-adopted), not an LLM guess | "How is your nutrition score not just Gemini making something up?" |

**Run order:** top to bottom.

## 0. Setup — Install & Imports

Only installs what Colab doesn't already have. Deliberately no forced TensorFlow/scikit-learn upgrade -- Colab's preinstalled versions already match what trained your model.

In [ ]:
!pip install -q google-genai easyocr gdown requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pickle
import re
import json
import time
import requests
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
import gdown
import easyocr
from google import genai
from getpass import getpass


### Load model + preprocessors
File IDs already filled in.

In [ ]:
MODEL_FILE_ID = "1kM577z4OiSRBtOsZ_HqB2H8gyMusxOMG"
PREPROCESSORS_FILE_ID = "1uwO1n9JE4mBl5S4tTNAmZSvh8d9Qf8T1"

gdown.download(id=MODEL_FILE_ID, output="safescan_classifier.h5", quiet=False)
gdown.download(id=PREPROCESSORS_FILE_ID, output="preprocessors.pkl", quiet=False)

classifier = load_model("safescan_classifier.h5")

with open("preprocessors.pkl", "rb") as f:
    preproc = pickle.load(f)

vectorizer     = preproc["vectorizer"]
scaler_img     = preproc["scaler_img"]
scaler_text    = preproc["scaler_text"]
label_to_int   = preproc["label_to_int"]
unique_labels  = preproc["unique_labels"]
int_to_label   = {v: k for k, v in label_to_int.items()}

print("Model + preprocessors loaded. Classes:", unique_labels)


Downloading...
From: https://drive.google.com/uc?id=1kM577z4OiSRBtOsZ_HqB2H8gyMusxOMG
To: /content/safescan_classifier.h5
100%|██████████| 17.8M/17.8M [00:00<00:00, 52.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1uwO1n9JE4mBl5S4tTNAmZSvh8d9Qf8T1
To: /content/preprocessors.pkl
100%|██████████| 80.3k/80.3k [00:00<00:00, 57.4MB/s]


Model + preprocessors loaded. Classes: ['ALMONDS_NUTS', 'BISCUITS', 'BUTTER', 'CEREAL', 'CHEESE', 'CHIPS_SNACKS', 'CHOCOLATE_CANDY', 'HONEY', 'MILK', 'NOODLES_PASTA', 'OIL', 'PICKLE', 'PULSES_DAL', 'RICE', 'SALT', 'SAUCE_KETCHUP', 'SOFTDRINK_JUICE', 'SPICES', 'SUGAR', 'TEA', 'WATER']


In [ ]:
GEMINI_API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=GEMINI_API_KEY)

# Rolling alias -- avoids repeat breakage when Google retires specific model
# versions. Free tier, no billing required.
GEMINI_MODEL = "gemini-flash-latest"

print("Gemini client ready.")


Paste your Gemini API key: ··········
Gemini client ready.


In [ ]:
def call_gemini_with_retry(prompt, retries=6, base_delay=4):
    """
    Calls Gemini with exponential backoff on transient errors (like 503
    UNAVAILABLE under free-tier demand spikes). 4s -> 8s -> 16s -> 32s -> 60s,
    ~3 minutes total worst-case wait before giving up.
    """
    last_error = None
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        except Exception as e:
            last_error = e
            if attempt < retries - 1:
                delay = min(base_delay * (2 ** attempt), 60)
                print(f"Gemini busy (attempt {attempt + 1}/{retries}), retrying in {delay}s...")
                time.sleep(delay)
    raise last_error


In [ ]:
# EDIT THIS -- the allergens this particular user is avoiding
ALLERGY_PROFILE = ["peanuts", "milk", "gluten"]


### OCR extraction — identical to Notebook 3

In [ ]:
ocr_reader = easyocr.Reader(['en'], gpu=True)

def clean_text(raw_text):
    text = raw_text.lower()
    text = re.sub(r'[^a-z0-9\s%.,]', ' ', text)
    tokens = [t for t in text.split() if len(t) >= 2]
    return " ".join(tokens)

def extract_ocr_text(image_path):
    results = ocr_reader.readtext(image_path, detail=0)
    raw_text = " ".join(results)
    return clean_text(raw_text)


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

### Classification — front image only, matches Notebook 2's single concatenated (2048+500)-dim input.

`get_fused_features` / `get_softmax_probs` expose the raw probability vector (not just top-1), which Part A needs for calibration.

In [ ]:
resnet_base = ResNet50(weights="imagenet", include_top=False, pooling="avg")

def get_image_features(image_path):
    img = keras_image.load_img(image_path, target_size=(224, 224))
    x = keras_image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    return resnet_base.predict(x, verbose=0)  # shape (1, 2048)

def get_fused_features(front_image_path, front_ocr_text):
    img_feat = get_image_features(front_image_path)
    img_feat_scaled = scaler_img.transform(img_feat)

    text_feat = vectorizer.transform([front_ocr_text]).toarray()
    text_feat_scaled = scaler_text.transform(text_feat)

    return np.concatenate([img_feat_scaled, text_feat_scaled], axis=1)  # shape (1, 2548)

def get_softmax_probs(fused_features):
    return classifier.predict(fused_features, verbose=0)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


---
## Part A — Confidence Calibration (Mondrian Conformal Prediction)

Raw softmax confidence isn't statistically guaranteed. Mondrian conformal prediction calibrates it **per class**, using a held-out calibration set, so a "90% confidence set" actually contains the true label ~90% of the time. Ties to your own lit survey (Paper 6, Review-1 table).

In [ ]:
def compute_nonconformity_scores(probs, true_label_ints):
    true_label_ints = np.asarray(true_label_ints)
    n = len(true_label_ints)
    prob_true_class = probs[np.arange(n), true_label_ints]
    return 1.0 - prob_true_class


def calibrate_mondrian(probs_cal, y_cal_ints, unique_labels, label_to_int):
    scores = compute_nonconformity_scores(probs_cal, y_cal_ints)
    per_class_scores = {}
    for label in unique_labels:
        cls_int = label_to_int[label]
        mask = np.asarray(y_cal_ints) == cls_int
        per_class_scores[label] = np.sort(scores[mask]) if mask.sum() > 0 else np.array([])
    return per_class_scores


def predict_with_confidence_set(probs_new, per_class_scores, unique_labels,
                                  label_to_int, int_to_label, alpha=0.1):
    prediction_set = []
    for label in unique_labels:
        cls_int = label_to_int[label]
        class_scores = per_class_scores.get(label, np.array([]))
        if len(class_scores) == 0:
            continue
        threshold_idx = int(np.ceil((1 - alpha) * (len(class_scores) + 1))) - 1
        threshold_idx = min(threshold_idx, len(class_scores) - 1)
        threshold = class_scores[threshold_idx]
        nonconformity = 1.0 - probs_new[cls_int]
        if nonconformity <= threshold:
            prediction_set.append(label)

    top1_int = int(np.argmax(probs_new))
    top1_label = int_to_label[top1_int]
    top1_conf = float(probs_new[top1_int])
    return prediction_set, top1_label, top1_conf


In [ ]:
import gdown
import os, random, shutil

CROPPED_ZIP_FILE_ID = "1wjrNpehRDkUWAqzOBGWhumtM5cAzTpku"

gdown.download(id=CROPPED_ZIP_FILE_ID, output="/content/iitpatna_cropped.zip", quiet=False)
!unzip -q /content/iitpatna_cropped.zip -d /content/iitpatna_cropped_full

print("Extracted contents:")
!ls /content/iitpatna_cropped_full

def build_calibration_set(calibration_root):
    """
    calibration_root: folder with one subfolder per category, e.g.
        calibration_root/BISCUITS/img1.jpg, ...
    Subfolder names must exactly match unique_labels.
    """
    probs_list = []
    y_list = []
    for label in unique_labels:
        folder = os.path.join(calibration_root, label)
        if not os.path.isdir(folder):
            print(f"WARNING: no folder found for '{label}', skipping")
            continue
        img_files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"{label}: {len(img_files)} calibration images")
        for img_file in img_files:
            img_path = os.path.join(folder, img_file)
            ocr_text = extract_ocr_text(img_path)
            fused = get_fused_features(img_path, ocr_text)
            probs = get_softmax_probs(fused)[0]
            probs_list.append(probs)
            y_list.append(label_to_int[label])
    probs_cal = np.array(probs_list)
    y_cal_ints = np.array(y_list)
    print(f"\nTotal calibration samples: {len(y_cal_ints)}")
    return probs_cal, y_cal_ints

IMAGES_PER_CLASS = 20
random.seed(42)
CALIBRATION_ROOT = "/content/calibration_data"
os.makedirs(CALIBRATION_ROOT, exist_ok=True)

source_root = "/content/iitpatna_cropped_full/content/iitpatna_cropped"

for label in unique_labels:
    src_folder = os.path.join(source_root, label)
    dst_folder = os.path.join(CALIBRATION_ROOT, label)
    os.makedirs(dst_folder, exist_ok=True)
    if not os.path.isdir(src_folder):
        print(f"WARNING: no source folder for '{label}' at {src_folder}")
        continue
    all_imgs = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample = random.sample(all_imgs, min(IMAGES_PER_CLASS, len(all_imgs)))
    for f in sample:
        shutil.copy(os.path.join(src_folder, f), os.path.join(dst_folder, f))
    print(f"{label}: sampled {len(sample)} images")

probs_cal, y_cal_ints = build_calibration_set(CALIBRATION_ROOT)
per_class_scores = calibrate_mondrian(probs_cal, y_cal_ints, unique_labels, label_to_int)

print("\nCalibration complete. Samples per class used:")
for label, scores in per_class_scores.items():
    print(f"  {label}: {len(scores)}")

Downloading...
From (original): https://drive.google.com/uc?id=1wjrNpehRDkUWAqzOBGWhumtM5cAzTpku
From (redirected): https://drive.google.com/uc?id=1wjrNpehRDkUWAqzOBGWhumtM5cAzTpku&confirm=t&uuid=d666cb65-3871-45c7-82ed-dca5a824284e
To: /content/iitpatna_cropped.zip
100%|██████████| 267M/267M [00:00<00:00, 270MB/s]


replace /content/iitpatna_cropped_full/content/iitpatna_cropped/PICKLE/PICKLE_0197.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Extracted contents:
content
ALMONDS_NUTS: sampled 20 images
BISCUITS: sampled 20 images
BUTTER: sampled 20 images
CEREAL: sampled 20 images
CHEESE: sampled 20 images
CHIPS_SNACKS: sampled 20 images
CHOCOLATE_CANDY: sampled 20 images
HONEY: sampled 20 images
MILK: sampled 20 images
NOODLES_PASTA: sampled 20 images
OIL: sampled 20 images
PICKLE: sampled 20 images
PULSES_DAL: sampled 20 images
RICE: sampled 20 images
SALT: sampled 20 images
SAUCE_KETCHUP: sampled 20 images
SOFTDRINK_JUICE: sampled 20 images
SPICES: sampled 20 images
SUGAR: sampled 20 images
TEA: sampled 20 images
WATER: sampled 20 images
ALMONDS_NUTS: 20 calibration images


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

BISCUITS: 20 calibration images


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

...

TEA: 20 calibration images


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume

WATER: 20 calibration images


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argume


Total calibration samples: 420

Calibration complete. Samples per class used:
  ALMONDS_NUTS: 20
  BISCUITS: 20
  BUTTER: 20
  CEREAL: 20
  CHEESE: 20
  CHIPS_SNACKS: 20
  CHOCOLATE_CANDY: 20
  HONEY: 20
  MILK: 20
  NOODLES_PASTA: 20
  OIL: 20
  PICKLE: 20
  PULSES_DAL: 20
  RICE: 20
  SALT: 20
  SAUCE_KETCHUP: 20
  SOFTDRINK_JUICE: 20
  SPICES: 20
  SUGAR: 20
  TEA: 20
  WATER: 20


**To activate:** build `probs_cal` / `y_cal_ints` from a held-out labeled slice of your Notebook 2 data, then `calibrate_mondrian(...)` and pass the result into `run_safescan_v2(per_class_scores=...)`. Without it, falls back to plain top-1 confidence -- pipeline still runs.

---
## Part B — RAG Grounding for Allergen Detection

Retrieves real ingredient/allergen data from Open Food Facts for the predicted category, and includes it alongside the OCR text in the allergen prompt -- Gemini now cross-checks two sources instead of reasoning from OCR alone.

In [ ]:
OFF_SEARCH_URL = "https://world.openfoodfacts.org/cgi/search.pl"

def fetch_openfoodfacts_grounding(product_query, max_products=3):
    """
    Returns (grounding_text, hit_found). Log hit_found across test runs -- a
    frequent False for Indian products quantifies the panel's own
    'barcode database gap' point.
    """
    params = {
        "search_terms": product_query, "search_simple": 1,
        "action": "process", "json": 1, "page_size": max_products,
    }
    try:
        resp = requests.get(OFF_SEARCH_URL, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"[Open Food Facts lookup failed: {e}]", False

    products = data.get("products", [])
    if not products:
        return f"[No Open Food Facts entries found for '{product_query}']", False

    lines = [f"Verified reference data for '{product_query}' (source: Open Food Facts):"]
    for p in products:
        name = p.get("product_name", "Unknown product")
        ingredients = p.get("ingredients_text", "").strip()
        allergens = p.get("allergens", "").strip()
        if not ingredients and not allergens:
            continue
        lines.append(f"- {name}: ingredients=\"{ingredients[:300]}\" | listed_allergens=\"{allergens}\"")

    if len(lines) == 1:
        return f"[Open Food Facts had matches for '{product_query}' but no usable ingredient data]", False

    return "\n".join(lines), True


In [ ]:
def check_allergens_grounded(back_ocr_text, allergy_profile, grounding_context):
    prompt = f"""You are a food safety assistant. Below is raw OCR text extracted
from a grocery product's ingredient label (it may contain noise/typos from OCR).

OCR TEXT:
{back_ocr_text}

Below is VERIFIED reference data retrieved from Open Food Facts for a similar
product -- use it to cross-check the OCR reading, but the OCR text is the primary
source since it's the actual product being scanned:

{grounding_context}

USER'S ALLERGENS TO AVOID: {", ".join(allergy_profile)}

Task:
1. List any ingredients (from OCR text and/or the reference data) that conflict
   with the user's allergens.
2. If none are found, clearly say the product appears safe based on available text,
   but note that OCR can miss text, so the user should still check the physical label.
3. State explicitly whether your verdict relied on the OCR text, the reference data,
   or both.
4. Keep the explanation short -- 2 to 4 sentences, plain language, no medical jargon.

Respond in this exact format:
VERDICT: <SAFE / CONFLICT FOUND / UNCERTAIN>
DETAILS: <your explanation>"""

    response = call_gemini_with_retry(prompt)
    return response.text


---
## Part C — Standardized Nutrition Scoring (Nutri-Score)

**The problem this fixes:** right now, asking Gemini for "a health score out of 10" is Gemini inventing a number from its own judgment -- a panelist can fairly call that unfounded, the same "just calling the API" critique that applies to allergens without RAG.

**The fix:** split the job into two parts, each doing what it's actually good at:
1. **Gemini's job -- structured extraction only.** Read the noisy OCR text and pull out the actual nutrient numbers (energy, sugar, saturated fat, sodium, fibre, protein) into clean fields. This is text understanding, which LLMs are genuinely good at, and it's not a judgment call.
2. **A deterministic formula's job -- the actual scoring.** Feed those numbers into the **Nutri-Score algorithm** (published by Sante publique France, adopted across multiple EU countries, closest thing to a peer-reviewed, publicly documented nutrient profiling standard). It scores based on fixed, publicly documented thresholds -- no LLM judgment involved in the final grade.

This mirrors what RAG grounding does for allergens: replace "trust the LLM" with "trust a verified, external, published source" -- just applied to nutrition instead of ingredients.

**Caveat to be upfront about with the panel:** Nutri-Score is a *general* nutrient profiling algorithm, not an India-specific one (FSSAI's own Indian system is still in development and not finalized). Using it is defensible as "a real, published, internationally-adopted standard" rather than an invented one -- but it's fair to note as a limitation, and a natural direction for future work once an Indian-specific standard exists.

In [ ]:
def extract_nutrient_facts(back_ocr_text):
    """
    Gemini's ONLY job here is structured extraction -- pulling numeric nutrient
    values out of noisy OCR text into clean JSON fields. No scoring or judgment.
    """
    prompt = f"""You are extracting nutrition facts from raw OCR text of a grocery
product's nutrition panel (OCR may contain noise/typos).

OCR TEXT:
{back_ocr_text}

Extract these values per 100g/100ml if present in the text (use null if not found
or not determinable -- do not guess):
- energy_kcal_100g (number)
- sugars_g_100g (number)
- saturated_fat_g_100g (number)
- sodium_mg_100g (number) -- if only "salt" is given in grams, convert with
  sodium_mg = salt_g * 400
- fibre_g_100g (number)
- protein_g_100g (number)
- fruit_veg_nuts_pct (number, % fruit/vegetable/nut/pulse content if explicitly
  stated, else null)

Respond with ONLY a valid JSON object with exactly these seven keys. No other text,
no markdown code fences, no explanation."""

    response = call_gemini_with_retry(prompt)
    raw = response.text.strip()
    # Strip accidental code fences if the model adds them anyway
    raw = re.sub(r'^```(json)?|```$', '', raw.strip(), flags=re.MULTILINE).strip()

    expected_keys = ["energy_kcal_100g", "sugars_g_100g", "saturated_fat_g_100g",
                      "sodium_mg_100g", "fibre_g_100g", "protein_g_100g", "fruit_veg_nuts_pct"]
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = {}

    return {k: parsed.get(k) for k in expected_keys}


In [ ]:
# Published Nutri-Score thresholds (Sante publique France, 2017 algorithm,
# "general foods" category). Each list gives the upper bound for points 0..N-1;
# a value above the last threshold scores the maximum point value.
ENERGY_THRESHOLDS_KJ = [335, 670, 1005, 1340, 1675, 2010, 2345, 2680, 3015, 3350]   # 0-10 pts
SUGAR_THRESHOLDS_G    = [4.5, 9, 13.5, 18, 22.5, 27, 31, 36, 40, 45]                # 0-10 pts
SATFAT_THRESHOLDS_G   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]                             # 0-10 pts
SODIUM_THRESHOLDS_MG  = [90, 180, 270, 360, 450, 540, 630, 720, 810, 900]           # 0-10 pts
FIBRE_THRESHOLDS_G    = [0.9, 1.9, 2.8, 3.7, 4.7]                                   # 0-5 pts
PROTEIN_THRESHOLDS_G  = [1.6, 3.2, 4.8, 6.4, 8.0]                                   # 0-5 pts


def _lookup(value, thresholds):
    for i, t in enumerate(thresholds):
        if value <= t:
            return i
    return len(thresholds)


def _fruit_veg_nuts_points(pct):
    if pct is None:
        return 0
    if pct <= 40:
        return 0
    elif pct <= 60:
        return 1
    elif pct <= 80:
        return 2
    else:
        return 5


def compute_nutriscore(nutrients):
    """
    Implements the published Nutri-Score algorithm (general foods category).
    Missing fields default to 0 for the calculation (best-case assumption) and
    are flagged separately via `data_complete` / `missing_fields` -- the score
    is explicitly marked partial/indicative when data is incomplete, rather than
    silently presenting an uncertain number as definitive.
    """
    core_fields = ["energy_kcal_100g", "sugars_g_100g", "saturated_fat_g_100g",
                    "sodium_mg_100g", "fibre_g_100g", "protein_g_100g"]
    missing = [k for k in core_fields if nutrients.get(k) is None]

    energy_kcal = nutrients.get("energy_kcal_100g") or 0
    energy_kj   = energy_kcal * 4.184
    sugars      = nutrients.get("sugars_g_100g") or 0
    sat_fat     = nutrients.get("saturated_fat_g_100g") or 0
    sodium      = nutrients.get("sodium_mg_100g") or 0
    fibre       = nutrients.get("fibre_g_100g") or 0
    protein     = nutrients.get("protein_g_100g") or 0
    fruit_pct   = nutrients.get("fruit_veg_nuts_pct")

    energy_pts = _lookup(energy_kj, ENERGY_THRESHOLDS_KJ)
    sugar_pts  = _lookup(sugars, SUGAR_THRESHOLDS_G)
    satfat_pts = _lookup(sat_fat, SATFAT_THRESHOLDS_G)
    sodium_pts = _lookup(sodium, SODIUM_THRESHOLDS_MG)
    negative_points = energy_pts + sugar_pts + satfat_pts + sodium_pts

    fibre_pts   = _lookup(fibre, FIBRE_THRESHOLDS_G)
    protein_pts = _lookup(protein, PROTEIN_THRESHOLDS_G)
    fruit_pts   = _fruit_veg_nuts_points(fruit_pct)

    # Standard Nutri-Score rule: protein points only count if negative points < 11,
    # OR the fruit/veg/nuts score is already maxed out (5).
    if negative_points < 11 or fruit_pts == 5:
        positive_points = fibre_pts + protein_pts + fruit_pts
    else:
        positive_points = fibre_pts + fruit_pts

    final_score = negative_points - positive_points

    if final_score <= -1:
        grade = "A"
    elif final_score <= 2:
        grade = "B"
    elif final_score <= 10:
        grade = "C"
    elif final_score <= 18:
        grade = "D"
    else:
        grade = "E"

    return {
        "final_score": final_score,
        "grade": grade,
        "negative_points": negative_points,
        "positive_points": positive_points,
        "breakdown": {
            "energy_pts": energy_pts, "sugar_pts": sugar_pts,
            "satfat_pts": satfat_pts, "sodium_pts": sodium_pts,
            "fibre_pts": fibre_pts, "protein_pts": protein_pts, "fruit_pts": fruit_pts,
        },
        "data_complete": len(missing) == 0,
        "missing_fields": missing,
    }


def get_standardized_nutrition_score(back_ocr_text):
    """Full pipeline: Gemini extracts numbers, compute_nutriscore() grades them."""
    nutrients = extract_nutrient_facts(back_ocr_text)
    result = compute_nutriscore(nutrients)
    return nutrients, result


### Gemini's original freeform nutrition score — kept for comparison
Useful to show both side by side in the demo: Gemini's plain-language read vs. the standardized grade.

In [ ]:
def get_nutrition_score(back_ocr_text):
    prompt = f"""You are a nutrition assistant. Below is raw OCR text extracted from
a grocery product's nutrition facts panel (it may contain noise/typos from OCR).

OCR TEXT:
{back_ocr_text}

Task:
1. If nutrition facts (calories, sugar, fat, sodium, etc.) are identifiable in the text,
   give the product a health score out of 10 (10 = very healthy, 1 = very unhealthy).
2. If no nutrition facts are visible in the OCR text, say so clearly instead of guessing.
3. Give a 2 to 3 sentence plain-language reason for the score.

Respond in this exact format:
SCORE: <X/10 or "Not available">
REASON: <your explanation>"""

    response = call_gemini_with_retry(prompt)
    return response.text


---
## Merged Pipeline: `run_safescan_v2()`

Combines calibrated classification (Part A) + RAG-grounded allergen check (Part B) + both nutrition views (Part C: Gemini's freeform read alongside the standardized Nutri-Score grade).

In [ ]:
def run_safescan_v2(front_image_path, back_image_path, allergy_profile=ALLERGY_PROFILE,
                     per_class_scores=None, alpha=0.1):
    print(f"Front image: {front_image_path}")
    print(f"Back image : {back_image_path}")

    front_ocr_text = extract_ocr_text(front_image_path)
    fused = get_fused_features(front_image_path, front_ocr_text)
    probs = get_softmax_probs(fused)[0]

    if per_class_scores is not None:
        pred_set, top1_label, top1_conf = predict_with_confidence_set(
            probs, per_class_scores, unique_labels, label_to_int, int_to_label, alpha=alpha
        )
        category_line = (f"Category (top-1): {top1_label}  (raw confidence: {top1_conf:.1%})\n"
                          f"Calibrated {(1-alpha)*100:.0f}% confidence set: {pred_set}")
        category_for_lookup = top1_label
    else:
        top1_int = int(np.argmax(probs))
        top1_label = int_to_label[top1_int]
        top1_conf = float(probs[top1_int])
        category_line = f"Category: {top1_label}  (confidence: {top1_conf:.1%})  [uncalibrated]"
        category_for_lookup = top1_label

    back_ocr_text = extract_ocr_text(back_image_path)

    grounding_context, hit_found = fetch_openfoodfacts_grounding(category_for_lookup)
    allergen_result = check_allergens_grounded(back_ocr_text, allergy_profile, grounding_context)

    gemini_nutrition_text = get_nutrition_score(back_ocr_text)
    nutrients, nutriscore_result = get_standardized_nutrition_score(back_ocr_text)

    print("\n" + "=" * 55)
    print("SAFESCAN REPORT (v2 -- calibration + RAG + Nutri-Score)")
    print("=" * 55)
    print(category_line)
    print(f"Front OCR text (raw): {front_ocr_text[:150]}{'...' if len(front_ocr_text) > 150 else ''}")
    print(f"Back OCR text (raw) : {back_ocr_text[:200]}{'...' if len(back_ocr_text) > 200 else ''}")
    print(f"Open Food Facts grounding: {'FOUND' if hit_found else 'NOT FOUND'} for '{category_for_lookup}'")
    print("-" * 55)
    print("ALLERGEN CHECK (RAG-grounded)")
    print(allergen_result)
    print("-" * 55)
    print("NUTRITION -- Gemini's freeform read")
    print(gemini_nutrition_text)
    print("-" * 55)
    print("NUTRITION -- Standardized Nutri-Score (published algorithm)")
    print(f"Grade: {nutriscore_result['grade']}  (score: {nutriscore_result['final_score']}, "
          f"negative pts: {nutriscore_result['negative_points']}, positive pts: {nutriscore_result['positive_points']})")
    if not nutriscore_result["data_complete"]:
        print(f"NOTE: incomplete data -- missing fields treated as 0: {nutriscore_result['missing_fields']}")
    print(f"Extracted nutrients: {nutrients}")
    print("=" * 55)

    return {
        "category": category_for_lookup,
        "confidence": top1_conf,
        "front_ocr_text": front_ocr_text,
        "back_ocr_text": back_ocr_text,
        "grounding_hit": hit_found,
        "allergen_result": allergen_result,
        "gemini_nutrition_text": gemini_nutrition_text,
        "nutrients": nutrients,
        "nutriscore_result": nutriscore_result,
    }


### Upload an image and test — front, then back.

In [ ]:
from google.colab import files

print("Upload the FRONT image:")
front_uploaded = files.upload()
front_image_path = list(front_uploaded.keys())[0]

print("\nUpload the BACK image:")
back_uploaded = files.upload()
back_image_path = list(back_uploaded.keys())[0]

result = run_safescan_v2(
    front_image_path, back_image_path,
    allergy_profile=ALLERGY_PROFILE,
    per_class_scores=per_class_scores,   # swap in your calibrated per_class_scores once Part A is wired up
    alpha=0.1
)


Upload the FRONT image:


Saving IMG_7334.jpg to IMG_7334 (1).jpg

Upload the BACK image:


Saving IMG_20190419_171130.jpg to IMG_20190419_171130 (1).jpg
Front image: IMG_7334 (1).jpg
Back image : IMG_20190419_171130 (1).jpg


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Gemini busy (attempt 1/6), retrying in 4s...
Gemini busy (attempt 2/6), retrying in 8s...

SAFESCAN REPORT (v2 -- calibration + RAG + Nutri-Score)
Category (top-1): BISCUITS  (raw confidence: 99.4%)
Calibrated 90% confidence set: []
Front OCR text (raw): artificial flavouring subs inacool hygienic and day place transfer contente to cleah airtight contnner once caened britannia nutci choice digestive hi...
Back OCR text (raw) : 8812042800, 100 years of britahhia mail feedbackobritindiacom nutrition 1o0g product ac0n information vgredients carbohydrates 689 rmfaved wheat mlour whc le wheat of which sugars 14.59 edible veietab...
Open Food Facts grounding: NOT FOUND for 'BISCUITS'
-------------------------------------------------------
ALLERGEN CHECK (RAG-grounded)
VERDICT: CONFLICT FOUND
DETAILS: This product contains wheat ingredients (such as wheat flour and wheat bran), which contain gluten, as well as milk solids (OCR text "mck solids"), which conflict with your milk allergen. Becaus

---
## Notes / TODO

1. **Part A (real remaining gap)**: plug in a held-out labeled calibration split from Notebook 2's data -> `calibrate_mondrian(...)` -> pass into `run_safescan_v2(per_class_scores=...)`.
2. **Part C caveat**: Nutri-Score is a general, published EU standard -- not India-specific. Good talking point: "we grounded nutrition scoring in a real, published standard rather than an invented one; adopting an India-specific standard is a natural next step once FSSAI finalizes one."
3. **`fruit_veg_nuts_pct` will almost always be `null`** from OCR (packaging rarely states this explicitly) -- the algorithm defaults it to 0 points, which is the conservative (non-inflating) choice, not a flaw, but worth mentioning if asked.
4. **Open Food Facts coverage**: log `hit_found` across test runs -- a frequent `False` for Indian products quantifies the panel's "barcode database gap" point.
5. **API key**: rotate immediately if ever pasted in plain text anywhere.